# Capstone project : Autonomous Trader

In [1]:
from dotenv import load_dotenv
from agents import Agent,Runner,trace,OpenAIChatCompletionsModel,Tool
from agents.mcp import MCPServerStdio
import os
from IPython.display import display,Markdown
from datetime import datetime
from openai import AsyncOpenAI
from accounts import Account
from accounts_client import read_accounts_resource,read_strategy_resource

load_dotenv(override=True)

True

In [2]:
polygon_api_key=os.getenv("POLYGON_API_KEY")

market_mcp={"command":"uv","args":["run","market_server.py"]}

In [3]:
trader_mcp_server_params=[
    {"command":"uv","args":["run","accounts_server.py"]},
    {"command":"uv","args":["run","push_server.py"]},
    market_mcp
]

## And now our reseacher

In [4]:
tavily_env={"TAVILY_API_KEY":os.getenv("TAVILY_API")}

researcher_mcp_server_params=[
    {"command":"uvx","args":["mcp-server-fetch"]},
    {"command": "npx","args": ["-y", "tavily-mcp@latest"],"env":tavily_env}
]

## Now creating the MCMCPServerStdiofor each

In [5]:
researcher_mcp_servers=[MCPServerStdio(params=params,client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers=[MCPServerStdio(params=params,client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers=trader_mcp_servers+researcher_mcp_servers

## Lets make our reseacher agent

In [6]:
import os
groq_api_key = os.getenv('GROQ_API_KEY')
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
gpt_oss_120 = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)
gpt_oss_20 = OpenAIChatCompletionsModel(model="openai/gpt-oss-20b", openai_client=groq_client)
llama = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)
scout= OpenAIChatCompletionsModel(model="meta-llama/llama-4-scout-17b-16e-instruct", openai_client=groq_client)

In [7]:
async def get_researcher(mcp_servers)->Agent:
    instructions=f"""You are a financial researcher. You are able to search the web for interesting financial news,
    look for the possible trading opportunities, and help with research.
    Based on the request, you carry out the necessary research and repond with your findings.
    Take time to make multiple searches to get a comprehensive view, and then summarize your findings.
    If there in not a specific request , then just respond with the investment opportunities based on searching latest news.
    The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

    RESEARCH RULES:

- Use as few searches as necessary to answer the question accurately.
- Start with broad searches and only perform follow-up searches when needed.
- Never repeatedly search for the same information.
- Limit yourself to a maximum of 5-8 searches for broad research tasks.
- For specific questions, prefer 2-4 focused searches.
- Read only the most relevant sources.
- Ignore duplicate articles covering the same event.
- Summarize information immediately instead of retaining large amounts of raw text.
- Never copy long passages from articles.
- Extract only key facts, numbers, dates, catalysts, risks, and conclusions.
- Keep intermediate reasoning concise.
- When enough evidence has been collected, stop searching and produce the answer.
    """

    researcher=Agent(
    name="Researcher",
    instructions=instructions,
    model=llama,
    mcp_servers=mcp_servers
    )
    return researcher

In [8]:
async def get_researcher_tool(mcp_servers)->Tool:
    researcher=await get_researcher(mcp_servers)
    return researcher.as_tool(
     tool_name="Researcher",
     tool_description="This tool researches online for news and opportunities,\
     either baswed on your specific request to look into a certain stock,\
     or generally for natoble financial news and opportunities.\
     Describe what kind of research are you looking for ."
    )
    

In [9]:
researcher_question="Whats latest news on amazon"

for server in researcher_mcp_servers:
    await server.connect()

researcher=await get_researcher(researcher_mcp_servers)

with trace("Reseracher"):
    result=await Runner.run(researcher,researcher_question,max_turns=10)

In [10]:
display(Markdown(result.final_output))

Based on the latest news, Amazon has recently announced a multibillion-dollar investment in a new data center campus in Missouri, which is expected to create over 400 jobs and generate hundreds of millions in tax revenue. Additionally, Amazon has launched new EC2 instances powered by AWS Graviton5 processors and has added an AI traffic monetization capability to AWS WAF. However, there are also reports of Amazon delaying union contracts and exploiting workers at its Staten Island facility. The company's stock price has fallen after a strong earnings report, with chip demand and cloud growth being bright spots.

In [23]:
harsh_initial_strategy="You are a day trader who aggressively buys and sells shares based on the news and market conditions."
Account.get("Harsh").reset(harsh_initial_strategy)

display(Markdown(await read_accounts_resource("Harsh")))
display(Markdown(await read_strategy_resource("Harsh")))

{"name": "harsh", "balance": 10000.0, "strategy": "You are a day trader who aggressively buys and sells shares based on the news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-06-16 11:52:23", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

You are a day trader who aggressively buys and sells shares based on the news and market conditions.

## Create Our Trader Agent

In [24]:
agent_name="Harsh"

account_details=await read_accounts_resource(agent_name)
strategy=await read_strategy_resource(agent_name)

instructions = f"""
You are a trader that manages a portfolio of shares. Your name is {agent_name} and your account is under your name, {agent_name}.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
{strategy}
Your current holdings and balance is:
{account_details}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.
Please make use of these tools to manage your portfolio. Carry out trades as you see fit; do not wait for instructions or ask for confirmation.
"""

prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [25]:
print(instructions)


You are a trader that manages a portfolio of shares. Your name is Harsh and your account is under your name, Harsh.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
You are a day trader who aggressively buys and sells shares based on the news and market conditions.
Your current holdings and balance is:
{"name": "harsh", "balance": 10000.0, "strategy": "You are a day trader who aggressively buys and sells shares based on the news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-06-16 11:52:23", 10000.0], ["2026-06-16 11:52:31", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking

In [26]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="gpt-4o-mini",
)


with trace(agent_name):
    result=await Runner.run(trader,prompt,max_turns=20)

In [27]:
display(Markdown(result.final_output))

### Summary of Actions:

1. **Stock Price Checks**:
   - **Apple (AAPL)**: $296.42
   - **Tesla (TSLA)**: $411.15
   - **Alphabet (GOOGL)**: $399.76
   - **Microsoft (MSFT)**: $369.35

2. **Buy Transactions**:
   - **AAPL**: Bought **10 shares** for a bullish outlook based on overall market sentiment.
   - **TSLA**: Bought **5 shares** due to positive growth expectations in the electric vehicle sector.

3. **Sell Transactions Attempted**:
   - Attempted to sell **3 shares** of GOOGL and **2 shares** of MSFT, but these were unsuccessful as there were not enough shares held in your portfolio.

### Current Holdings:
- **AAPL**: 10 shares
- **TSLA**: 5 shares

### Account Balance:
- Current Balance: **$4970.01**

### Next Steps:
I will keep monitoring the market and consider further trades based on evolving news and trends. If you have any particular stocks you wish to explore or special strategies, just let me know!

In [28]:
account_details=await read_accounts_resource(agent_name)
print(account_details)

{"name": "harsh", "balance": 4970.0100999999995, "strategy": "You are a day trader who aggressively buys and sells shares based on the news and market conditions.", "holdings": {"AAPL": 10, "TSLA": 5}, "transactions": [{"symbol": "AAPL", "quantity": 10, "price": 297.01284000000004, "timestamp": "2026-06-16 11:53:23", "rationale": "Bullish trend in tech stocks and positive consumer sentiment."}, {"symbol": "TSLA", "quantity": 5, "price": 411.97229999999996, "timestamp": "2026-06-16 11:53:23", "rationale": "Strong production numbers and growth outlook for electric vehicles."}], "portfolio_value_time_series": [["2026-06-16 11:52:23", 10000.0], ["2026-06-16 11:52:31", 10000.0], ["2026-06-16 11:53:23", 9994.0716], ["2026-06-16 11:53:23", 9989.9601], ["2026-06-16 11:55:44", 9989.9601]], "total_portfolio_value": 9989.9601, "total_profit_loss": -10.039899999999761}
